In [12]:
from pathlib import Path
import sentencepiece as spm

In [13]:
train_model_dir = Path.cwd()
model_dir = train_model_dir / "tokenizers"
model_dir.mkdir(parents=True, exist_ok=True)

train_data_dir = train_model_dir.parents[1]/"data"/"sample_for_train_val_test"
train_data_en_path = train_data_dir / "train_en.txt"
train_data_zh_path = train_data_dir / "train_zh.txt"

zh_model_prefix = model_dir / "zh_model"
en_model_prefix = model_dir / "en_model"


In [14]:
spm.SentencePieceTrainer.train(input=train_data_zh_path, 
                               model_prefix=zh_model_prefix, 
                               vocab_size=10000,
                                model_type='bpe',
                                character_coverage=0.995,
                                pad_id=0,
                                unk_id=1,
                                bos_id=2,
                                eos_id=3)

spm.SentencePieceTrainer.train(input=train_data_en_path, 
                               model_prefix=en_model_prefix, 
                               vocab_size=10000,
                                model_type='bpe',
                                character_coverage=0.995,
                                pad_id=0,
                                unk_id=1,
                                bos_id=2,
                                eos_id=3)

True

In [15]:
# 加载已经训练好的中文和英文模型
zh_sp = spm.SentencePieceProcessor(
    model_file=str(model_dir / "zh_model.model"))
en_sp = spm.SentencePieceProcessor(
    model_file=str(model_dir / "en_model.model"))

text1 = "我喜欢学习机器。"
text2 = "I like to study machines."

# 分别输出 Token、Token ID 和解码结果
print("中文分词结果：", zh_sp.encode(text1, out_type=str))
print(type(zh_sp.encode(text1, out_type=str)))
print("英文分词结果：", en_sp.encode(text2,out_type=str))


中文分词结果： ['▁我', '喜', '欢', '学习', '机', '器', '。']
<class 'list'>
英文分词结果： ['▁I', '▁like', '▁to', '▁study', '▁mach', 'ines', '.']


In [16]:
zh_tokens = zh_sp.encode(text1, out_type=int)
en_tokens = en_sp.encode(text2,out_type=int)
print("中文分词结果对应的 Token ID：", zh_tokens)
print("英文分词结果对应的 Token ID：", en_tokens)

中文分词结果对应的 Token ID： [761, 1, 8524, 2159, 7933, 8260, 7823]
英文分词结果对应的 Token ID： [70, 1879, 35, 2360, 5473, 1024, 9962]


In [17]:
# 中文源句子
zh_text = "我喜欢学习机器。"
# 英文目标句子
en_text = "I like to study machines."
# 先分词，转换成 Token ID
zh_ids = zh_sp.encode(zh_text, out_type=int)
en_ids = en_sp.encode(en_text, out_type=int)
# 在序列开头和结尾分别添加 BOS、EOS
zh_ids = [zh_sp.bos_id()] + zh_ids + [zh_sp.eos_id()]
en_ids = [en_sp.bos_id()] + en_ids + [en_sp.eos_id()]

print("中文 ID：", zh_ids)
print("英文 ID：", en_ids)
print("中文序列长度：", len(zh_ids))
print("英文序列长度：", len(en_ids))

中文 ID： [2, 761, 1, 8524, 2159, 7933, 8260, 7823, 3]
英文 ID： [2, 70, 1879, 35, 2360, 5473, 1024, 9962, 3]
中文序列长度： 9
英文序列长度： 9


In [18]:
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, zh_path, en_path,
                 zh_sp, en_sp, max_len=1024):

        zh_lines = Path(zh_path).read_text(encoding="utf-8").splitlines()
        en_lines = Path(en_path).read_text(encoding="utf-8").splitlines()

        if len(zh_lines) != len(en_lines):
            raise ValueError("中英文行数不一致！")

        self.samples = []
        dropped = 0

        for zh, en in zip(zh_lines, en_lines):

            # 中文：文本 -> ID，并添加起止标记
            src_ids = ([zh_sp.bos_id()] + zh_sp.encode(zh.strip(), out_type=int)+ [zh_sp.eos_id()])
            # 英文：文本 -> ID，并添加起止标记
            tgt_ids = ([en_sp.bos_id()] + en_sp.encode(en.strip(), out_type=int)+ [en_sp.eos_id()])
            # 过长的句子对暂时不纳入本轮实验
            if len(src_ids) > max_len or len(tgt_ids) > max_len:
                dropped += 1
                continue

            self.samples.append((src_ids, tgt_ids))

        print(f"保留句子对：{len(self.samples)}")
        print(f"过滤句子对：{dropped}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [19]:
train_dataset = TranslationDataset(
    train_data_dir / "train_zh.txt",
    train_data_dir / "train_en.txt",
    zh_sp, en_sp,
    max_len=1024
)

val_dataset = TranslationDataset(
    train_data_dir / "val_zh.txt",
    train_data_dir / "val_en.txt",
    zh_sp, en_sp,
    max_len=1024
)

test_dataset = TranslationDataset(
    train_data_dir / "test_zh.txt",
    train_data_dir / "test_en.txt",
    zh_sp, en_sp,
    max_len=1024
)

保留句子对：449994
过滤句子对：6
保留句子对：25000
过滤句子对：0
保留句子对：25000
过滤句子对：0


In [20]:
def collate_translation(batch):
    # 1. 将列表转换为 PyTorch 的整数张量
    src_list = [torch.tensor(src, dtype=torch.long)for src, tgt in batch]
    tgt_list = [torch.tensor(tgt, dtype=torch.long)for src, tgt in batch]
    # 2. 分别补齐中文和英文序列
    src = pad_sequence(src_list,batch_first=True,padding_value=zh_sp.pad_id())
    tgt = pad_sequence(tgt_list,batch_first=True,padding_value=en_sp.pad_id())
    # 3. Decoder 输入与预测目标错开一个 Token
    decoder_input = tgt[:, :-1]
    labels = tgt[:, 1:].clone()
    # 4. 损失函数忽略英文目标中的 PAD
    labels[labels == en_sp.pad_id()] = -100
    # 5. 标记源句子和 Decoder 输入中的填充位置
    src_padding_mask = src.eq(zh_sp.pad_id())
    tgt_padding_mask = decoder_input.eq(en_sp.pad_id())
    # 6. 创建因果掩码，禁止 Decoder 偷看未来 Token
    target_len = decoder_input.shape[1]
    # true表示要被遮挡的，false表示可以看到的
    causal_mask = torch.triu(torch.ones(target_len, target_len,dtype=torch.bool),diagonal=1)

    return {
        "src": src,
        "decoder_input": decoder_input,
        "labels": labels,
        "src_padding_mask": src_padding_mask,
        "tgt_padding_mask": tgt_padding_mask,
        "causal_mask": causal_mask
    }

In [21]:
batch_size = 64
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_translation,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_translation,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_translation,
    num_workers=0
)

In [22]:
batch = next(iter(train_loader))

for name, tensor in batch.items():
    print(
        f"{name:20s}",
        "shape =", tuple(tensor.shape),
        "dtype =", tensor.dtype
    )

src                  shape = (64, 68) dtype = torch.int64
decoder_input        shape = (64, 87) dtype = torch.int64
labels               shape = (64, 87) dtype = torch.int64
src_padding_mask     shape = (64, 68) dtype = torch.bool
tgt_padding_mask     shape = (64, 87) dtype = torch.bool
causal_mask          shape = (87, 87) dtype = torch.bool
